# 垃圾分类视觉模型 — Step 1：高质量测试集锁定 + 防泄漏迁移学习基线

本 Notebook 针对 Kaggle **`sumn2u/garbage-classification-v2`**（10 类常见垃圾）设计，核心目标是先建立**绝对干净、高质量且零泄漏的评估基准**，再在 Google Colab GPU 上完成高精度两阶段训练与外部泛化摸底：

1. **先选定并物理隔离高质量测试集（Test Lock）**：计算全部图像的损坏状态、MD5、分辨率、宽高比、亮度/对比度、拉普拉斯清晰度与 64-bit 感知哈希（`pHash`），优先从清晰、无重复的独立簇中按类别抽样锁定 `test_locked/`，并将测试样本及其全部同簇/边界近重复样本从剩余数据池中**物理剔除**。
2. **防泄漏防火墙**：仅在剔除后的 `train_val_pool/` 内使用 `StratifiedGroupKFold` 划分训练/验证集，并通过跨集哈希与路径断言校验。
3. **两阶段 `timm` 迁移学习**：默认使用 `convnext_tiny.fb_in22k_ft_in1k_384`（AMP 混合精度 + 平方根平滑类别加权 Loss + Google Drive 断点续训）。
4. **全面诊断与外部 OOD 摸底**：支持 2-view TTA、归一化混淆矩阵、Top-16 高置信错例网格、Grad-CAM 热力图、**RealWaste** 零样本外部测试以及 **Gradio** 公网 Web 演示。

> **运行提示**：请在 Colab 菜单选择 `代码执行程序 -> 更改运行时类型 -> GPU（推荐 T4 或 L4）`，然后按顺序运行各个 Section。

## Section 1 — 环境检查、Google Drive 挂载与全局配置 (`CFG`)

In [ ]:
!pip -q install timm kagglehub imagehash grad-cam gradio scikit-learn opencv-python-headless seaborn

In [ ]:
import hashlib
import json
import math
import os
import random
import shutil
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from tqdm.auto import tqdm

# 挂载 Google Drive（若在非 Colab 环境或未授权则回退到本地目录）
IN_COLAB = False
DRIVE_MOUNTED = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
    DRIVE_MOUNTED = Path("/content/drive/MyDrive").exists()
except Exception as exc:
    print(f"[Info] 未挂载 Google Drive（使用本地存储）: {exc}")

BASE_WORK_DIR = Path("/content") if IN_COLAB else Path("./colab_work")
DRIVE_ROOT = (
    Path("/content/drive/MyDrive/garbage_cls_step1")
    if DRIVE_MOUNTED
    else (BASE_WORK_DIR / "artifacts_step1")
)

CFG = {
    "dataset_slug": "sumn2u/garbage-classification-v2",
    "expected_classes": [
        "battery", "biological", "cardboard", "clothes", "glass",
        "metal", "paper", "plastic", "shoes", "trash",
    ],
    # 主力高精度模型；若想先极速验证可改为 "tf_efficientnetv2_s.in21k_ft_in1k"
    "model_name": "convnext_tiny.fb_in22k_ft_in1k_384",
    "seed": 42,
    # pHash 近重复判定阈值（64-bit 汉明距离 <= 6 视为同源/近重复）
    "phash_dup_threshold": 6,
    # 高质量测试集锁定配置
    "test_ratio": 0.15,
    "min_test_per_class": 30,
    "min_test_res": 224,
    "min_aspect": 0.4,
    "max_aspect": 2.5,
    "min_laplacian_var": 30.0,
    "min_brightness": 25.0,
    "max_brightness": 235.0,
    "min_contrast": 15.0,
    "reuse_existing_test_lock": True,
    # Train / Val 分组切分（6 折取第 0 折 -> 约占剩余池 16.7%，约占总数据 14.2%）
    "val_folds": 6,
    # 两阶段训练超参
    "stage1_epochs": 3,
    "stage1_lr": 1e-3,
    "stage2_epochs": 15,
    "backbone_lr": 1e-4,
    "head_lr": 5e-4,
    "weight_decay": 1e-2,
    "batch_size": 32,
    "grad_accum_steps": 2,
    "num_workers": 2,
    "drop_path_rate": 0.2,
    "label_smoothing": 0.05,
    "early_stop_patience": 5,
    # Gradio 演示置信度拒识阈值
    "ood_conf_threshold": 0.60,
    # 隔离数据盘与持久化产物盘
    "clean_root": str(BASE_WORK_DIR / "garbage_v2_clean"),
    "drive_root": str(DRIVE_ROOT),
    "manifest_dir": str(DRIVE_ROOT / "manifests"),
    "ckpt_dir": str(DRIVE_ROOT / "checkpoints"),
    "eval_dir": str(DRIVE_ROOT / "eval_outputs"),
    "real_photos_dir": str(DRIVE_ROOT / "my_real_photos"),
}

for d_key in ("clean_root", "drive_root", "manifest_dir", "ckpt_dir", "eval_dir", "real_photos_dir"):
    Path(CFG[d_key]).mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | BF16 supported: {torch.cuda.is_bf16_supported()}")
print(f"Clean local workspace : {CFG['clean_root']}")
print(f"Persistent Drive root : {CFG['drive_root']}")

## Section 2 — 下载 Kaggle 数据集至 Colab 本地盘

In [ ]:
import kagglehub


def locate_class_root(download_dir: str | Path, expected_classes: list[str]) -> Path:
    """Recursively locate the directory that directly contains the expected class folders."""
    root = Path(download_dir)
    expected_set = set(expected_classes)
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    best_dir = None
    best_overlap = 0
    for cand in candidates:
        subdirs = {d.name for d in cand.iterdir() if d.is_dir()}
        overlap = len(subdirs & expected_set)
        if overlap == len(expected_set):
            return cand
        if overlap > best_overlap:
            best_overlap = overlap
            best_dir = cand
    if best_dir is not None and best_overlap >= max(2, len(expected_set) // 2):
        print(f"[Warning] 只匹配到 {best_overlap}/{len(expected_set)} 个类别目录: {best_dir}")
        return best_dir
    raise FileNotFoundError(f"在 {download_dir} 下未找到包含预期类别目录的路径: {expected_classes}")


raw_download_path = kagglehub.dataset_download(CFG["dataset_slug"])
RAW_DATA_ROOT = locate_class_root(raw_download_path, CFG["expected_classes"])
print(f"Kaggle raw path : {raw_download_path}")
print(f"Resolved root   : {RAW_DATA_ROOT}")

## Section 3 — 图像完整性体检、质量指标计算与向量化 `pHash` 聚类

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}
POPCOUNT_LUT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def compute_image_record(path_str: str, label: str) -> dict:
    """Verify image integrity, compute MD5, 64-bit pHash, and visual quality metrics."""
    path = Path(path_str)
    try:
        raw_bytes = path.read_bytes()
        md5_hex = hashlib.md5(raw_bytes).hexdigest()

        with Image.open(path) as img_verify:
            img_verify.verify()

        with Image.open(path) as img_raw:
            rgb_img = ImageOps.exif_transpose(img_raw).convert("RGB")
            width, height = rgb_img.size
            if width <= 0 or height <= 0:
                raise ValueError(f"Invalid image dimensions: {width}x{height}")

            phash_int = int(str(imagehash.phash(rgb_img)), 16)
            phash_hex = f"{phash_int:016x}"

            rgb_np = np.asarray(rgb_img, dtype=np.uint8)
            gray = cv2.cvtColor(rgb_np, cv2.COLOR_RGB2GRAY)
            laplacian_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
            brightness_mean = float(gray.mean())
            contrast_std = float(gray.std())

        return {
            "orig_path": str(path),
            "orig_name": path.name,
            "label": label,
            "status": "ok",
            "error": "",
            "md5": md5_hex,
            "phash_hex": phash_hex,
            "width": int(width),
            "height": int(height),
            "min_side": int(min(width, height)),
            "aspect_ratio": float(width / height),
            "laplacian_var": laplacian_var,
            "brightness_mean": brightness_mean,
            "contrast_std": contrast_std,
        }
    except Exception as exc:
        return {
            "orig_path": str(path),
            "orig_name": path.name,
            "label": label,
            "status": "corrupted",
            "error": repr(exc),
            "md5": "",
            "phash_hex": "",
            "width": 0,
            "height": 0,
            "min_side": 0,
            "aspect_ratio": 0.0,
            "laplacian_var": 0.0,
            "brightness_mean": 0.0,
            "contrast_std": 0.0,
        }


def scan_dataset(
    dataset_root: str | Path,
    expected_classes: list[str],
    max_workers: int = 8,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Scan all class subdirectories in parallel and separate valid vs corrupted images."""
    root = Path(dataset_root)
    tasks: list[tuple[str, str]] = []
    for cls_name in expected_classes:
        cls_dir = root / cls_name
        if not cls_dir.is_dir():
            raise FileNotFoundError(f"缺少类别目录: {cls_dir}")
        files = sorted(
            p for p in cls_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        )
        for fpath in files:
            tasks.append((str(fpath), cls_name))

    records: list[dict] = []
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(compute_image_record, p, lbl) for p, lbl in tasks]
        for fut in tqdm(futures, total=len(futures), desc="Scanning images"):
            records.append(fut.result())

    full_df = pd.DataFrame(records)
    valid_df = full_df[full_df["status"] == "ok"].reset_index(drop=True)
    corrupted_df = full_df[full_df["status"] != "ok"].reset_index(drop=True)
    return valid_df, corrupted_df


def _hex_to_uint64(hex_values) -> np.ndarray:
    return np.array([int(h, 16) for h in hex_values], dtype=np.uint64)


def min_hamming_between(
    query_hex_values,
    ref_hex_values,
    chunk_size: int = 512,
) -> np.ndarray:
    """Compute minimum 64-bit Hamming distance from each query pHash to any reference pHash."""
    q_arr = _hex_to_uint64(query_hex_values)
    r_arr = _hex_to_uint64(ref_hex_values)
    if len(q_arr) == 0:
        return np.empty(0, dtype=np.int32)
    if len(r_arr) == 0:
        return np.full(len(q_arr), 64, dtype=np.int32)

    min_dists = np.full(len(q_arr), 64, dtype=np.int32)
    for start in range(0, len(q_arr), chunk_size):
        end = min(start + chunk_size, len(q_arr))
        q_chunk = q_arr[start:end]
        row_min = np.full(len(q_chunk), 64, dtype=np.int32)
        for r_start in range(0, len(r_arr), chunk_size * 4):
            r_end = min(r_start + chunk_size * 4, len(r_arr))
            xor = q_chunk[:, None] ^ r_arr[None, r_start:r_end]
            dists = POPCOUNT_LUT[xor.view(np.uint8).reshape(len(q_chunk), r_end - r_start, 8)].sum(axis=-1)
            row_min = np.minimum(row_min, dists.min(axis=1))
        min_dists[start:end] = row_min
    return min_dists


def cluster_phash_vectorized(
    phash_hex_list,
    threshold: int = 6,
    chunk_size: int = 512,
) -> tuple[np.ndarray, np.ndarray]:
    """Cluster near-duplicate images via vectorized uint64 XOR + POPCOUNT_LUT + Union-Find."""
    arr = _hex_to_uint64(phash_hex_list)
    n = len(arr)
    if n == 0:
        return np.empty(0, dtype=np.int64), np.empty(0, dtype=np.int64)

    parent = np.arange(n, dtype=np.int64)
    rank = np.zeros(n, dtype=np.int32)

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = int(parent[x])
        return x

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        if rank[ra] == rank[rb]:
            rank[ra] += 1

    for i in range(0, n, chunk_size):
        end = min(i + chunk_size, n)
        sub = arr[i:end]
        tail = arr[i:]
        xor = sub[:, None] ^ tail[None, :]
        dists = POPCOUNT_LUT[xor.view(np.uint8).reshape(end - i, n - i, 8)].sum(axis=-1)

        row_offsets = np.arange(end - i)[:, None]
        col_offsets = np.arange(n - i)[None, :]
        upper_mask = col_offsets > row_offsets
        match_rows, match_cols = np.where((dists <= threshold) & upper_mask)
        for r_off, c_off in zip(match_rows.tolist(), match_cols.tolist()):
            union(i + r_off, i + c_off)

    roots = np.array([find(idx) for idx in range(n)], dtype=np.int64)
    _, cluster_ids, counts = np.unique(roots, return_inverse=True, return_counts=True)
    cluster_sizes = counts[cluster_ids]
    return cluster_ids.astype(np.int64), cluster_sizes.astype(np.int64)

In [ ]:
valid_df, corrupted_df = scan_dataset(RAW_DATA_ROOT, CFG["expected_classes"], max_workers=8)
cluster_ids, cluster_sizes = cluster_phash_vectorized(
    valid_df["phash_hex"].tolist(),
    threshold=CFG["phash_dup_threshold"],
    chunk_size=512,
)
valid_df["cluster_id"] = cluster_ids
valid_df["cluster_size"] = cluster_sizes

n_clusters = int(valid_df["cluster_id"].nunique())
n_dup_imgs = int((valid_df["cluster_size"] > 1).sum())
print(f"有效图像数       : {len(valid_df)}")
print(f"损坏图像数（剔除）: {len(corrupted_df)}")
print(f"pHash 唯一簇数量  : {n_clusters}（其中含近重复图的图像有 {n_dup_imgs} 张）")

if len(corrupted_df) > 0:
    corrupted_df.to_csv(Path(CFG["manifest_dir"]) / "corrupted_images.csv", index=False)

cls_counts = valid_df["label"].value_counts().reindex(CFG["expected_classes"])
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x=cls_counts.index, y=cls_counts.values, hue=cls_counts.index, legend=False, ax=ax)
ax.set_title("Garbage Classification v2 — Valid Image Count per Class")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=30)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2.0, p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(Path(CFG["eval_dir"]) / "eda_class_distribution.png", dpi=150)
plt.show()

## Section 4 — 高质量测试集选定与严格物理剔除

本步严格执行“**先选高质量测试集，再从数据池彻底剔除**”流程：
1. **高质量准入门槛**：短边 $\ge 224$、宽高比 $\in [0.4, 2.5]$、均值亮度 $\in [25, 235]$、对比度 $\ge 15$、拉普拉斯清晰度不低于 `max(30.0, 同类 P20 分位)`。
2. **优先级抽样**：优先从**高质量独立无重复簇（`cluster_size == 1`）**中随机抽样，其次从剩余高质量多图簇中每簇仅选 1 张最高分代表图，不足再兜底并告警。
3. **严格剔除与物理隔离**：将所有命中测试集的 `cluster_id` 成员以及跨边界汉明距离 $\le 6$ 的近重复图全部移出，剩余数据写入 `train_val_pool/`，测试图锁定写入 `test_locked/`，并记录 `test_lock_sha256.txt`。

In [ ]:
def _compute_quality_metrics(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    work = df.copy()
    cls_p20 = work.groupby("label")["laplacian_var"].transform(lambda s: float(np.percentile(s, 20)))
    lap_floor = np.maximum(cfg["min_laplacian_var"], cls_p20)

    work["is_high_quality"] = (
        (work["min_side"] >= cfg["min_test_res"])
        & (work["aspect_ratio"] >= cfg["min_aspect"])
        & (work["aspect_ratio"] <= cfg["max_aspect"])
        & (work["brightness_mean"] >= cfg["min_brightness"])
        & (work["brightness_mean"] <= cfg["max_brightness"])
        & (work["contrast_std"] >= cfg["min_contrast"])
        & (work["laplacian_var"] >= lap_floor)
    )

    sharp_norm = np.clip(np.log1p(work["laplacian_var"]) / np.log1p(1000.0), 0.0, 1.0)
    res_norm = np.clip(work["min_side"] / 512.0, 0.0, 1.0)
    aspect_balance = 1.0 - np.clip(np.abs(np.log(work["aspect_ratio"].clip(lower=1e-3))) / math.log(2.5), 0.0, 1.0)
    work["quality_score"] = 0.50 * sharp_norm + 0.30 * res_norm + 0.20 * aspect_balance
    return work


def _materialize_isolated_split(df: pd.DataFrame, target_split_dir: Path) -> pd.DataFrame:
    out = df.copy().reset_index(drop=True)
    if target_split_dir.exists():
        shutil.rmtree(target_split_dir)
    target_split_dir.mkdir(parents=True, exist_ok=True)

    isolated_paths: list[str] = []
    for idx, row in out.iterrows():
        cls_dir = target_split_dir / str(row["label"])
        cls_dir.mkdir(parents=True, exist_ok=True)
        src = Path(row["orig_path"])
        dst = cls_dir / f"{idx:06d}_{src.name}"
        try:
            os.link(src, dst)
        except OSError:
            shutil.copy2(src, dst)
        isolated_paths.append(str(dst))

    out["isolated_path"] = isolated_paths
    return out


def _compute_lock_sha256(test_df: pd.DataFrame) -> str:
    payload = "\n".join(sorted(f"{lbl}:{md5}" for lbl, md5 in zip(test_df["label"], test_df["md5"])))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def select_high_quality_test_and_exclude(
    df: pd.DataFrame,
    cfg: dict,
    clean_root: str | Path,
    manifest_dir: str | Path,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Select and lock a high-quality test set first, then strictly exclude it and all near-duplicates."""
    clean_path = Path(clean_root)
    manifest_path = Path(manifest_dir)
    manifest_path.mkdir(parents=True, exist_ok=True)

    work = _compute_quality_metrics(df, cfg)
    test_csv_path = manifest_path / "test_locked.csv"
    sha_path = manifest_path / "test_lock_sha256.txt"

    reused_test_indices: list[int] | None = None
    if cfg.get("reuse_existing_test_lock", True) and test_csv_path.exists():
        prev_test = pd.read_csv(test_csv_path)
        prev_pairs = set(zip(prev_test["label"].astype(str), prev_test["md5"].astype(str)))
        matched_rows = []
        used_pair = set()
        for idx, row in work.iterrows():
            pair = (str(row["label"]), str(row["md5"]))
            if pair in prev_pairs and pair not in used_pair:
                matched_rows.append(idx)
                used_pair.add(pair)
        if len(matched_rows) == len(prev_pairs) and len(matched_rows) > 0:
            candidate_test = work.loc[matched_rows]
            recomputed_sha = _compute_lock_sha256(candidate_test)
            expected_sha = sha_path.read_text(encoding="utf-8").strip() if sha_path.exists() else recomputed_sha
            if recomputed_sha == expected_sha:
                reused_test_indices = matched_rows
                print(f"[Lock Reuse] 复用已有高质量锁定测试集（{len(reused_test_indices)} 张，SHA256 校验一致）。")
            else:
                print("[Warning] 已有 test_lock_sha256 校验不一致，将重新抽样锁定测试集。")
        else:
            print("[Warning] 已有 test_locked.csv 与当前扫描数据未完全匹配，将重新抽样锁定测试集。")

    if reused_test_indices is not None:
        selected_indices = reused_test_indices
    else:
        selected_indices = []
        used_clusters: set[int] = set()
        for cls_idx, cls_name in enumerate(cfg["expected_classes"]):
            cls_df = work[work["label"] == cls_name]
            n_clusters = int(cls_df["cluster_id"].nunique())
            cap_n = max(1, int(n_clusters * 0.30))
            target_n = min(max(cfg["min_test_per_class"], int(round(n_clusters * cfg["test_ratio"]))), cap_n)
            rng = np.random.default_rng(int(cfg["seed"]) + cls_idx)

            cls_picked: list[int] = []

            # Priority 1: 高质量独立无重复簇（cluster_size == 1）
            p1_df = cls_df[
                cls_df["is_high_quality"]
                & (cls_df["cluster_size"] == 1)
                & (~cls_df["cluster_id"].isin(used_clusters))
            ]
            if len(p1_df) > 0:
                take_n = min(target_n, len(p1_df))
                chosen = rng.choice(p1_df.index.to_numpy(), size=take_n, replace=False).tolist()
                cls_picked.extend(chosen)
                used_clusters.update(work.loc[chosen, "cluster_id"].astype(int).tolist())

            # Priority 2: 剩余高质量多图簇（每簇仅取 1 张最高分代表图）
            if len(cls_picked) < target_n:
                need = target_n - len(cls_picked)
                p2_df = cls_df[
                    cls_df["is_high_quality"] & (~cls_df["cluster_id"].isin(used_clusters))
                ]
                if len(p2_df) > 0:
                    p2_rep = (
                        p2_df.sort_values("quality_score", ascending=False)
                        .drop_duplicates(subset=["cluster_id"], keep="first")
                    )
                    take_n = min(need, len(p2_rep))
                    chosen = p2_rep.index.to_numpy()[:take_n].tolist()
                    cls_picked.extend(chosen)
                    used_clusters.update(work.loc[chosen, "cluster_id"].astype(int).tolist())

            # Priority 3: 兜底（从剩余未用簇取最高质量代表图，并记录告警）
            if len(cls_picked) < target_n:
                need = target_n - len(cls_picked)
                p3_df = cls_df[~cls_df["cluster_id"].isin(used_clusters)]
                if len(p3_df) > 0:
                    p3_rep = (
                        p3_df.sort_values("quality_score", ascending=False)
                        .drop_duplicates(subset=["cluster_id"], keep="first")
                    )
                    take_n = min(need, len(p3_rep))
                    chosen = p3_rep.index.to_numpy()[:take_n].tolist()
                    cls_picked.extend(chosen)
                    used_clusters.update(work.loc[chosen, "cluster_id"].astype(int).tolist())
                    print(
                        f"[Warning] 类别 {cls_name} 高质量独立样本不足，已从普通质量簇兜底补足 {take_n} 张。"
                    )

            selected_indices.extend(cls_picked)

    test_raw_df = work.loc[selected_indices].copy()
    locked_clusters = set(test_raw_df["cluster_id"].astype(int).tolist())

    not_test_mask = ~work.index.isin(test_raw_df.index)
    same_cluster_mask = not_test_mask & work["cluster_id"].isin(locked_clusters)
    candidate_mask = not_test_mask & (~same_cluster_mask)

    boundary_dup_mask = pd.Series(False, index=work.index)
    if candidate_mask.any() and len(test_raw_df) > 0:
        cand_indices = work.index[candidate_mask].to_numpy()
        min_dists = min_hamming_between(
            work.loc[cand_indices, "phash_hex"].tolist(),
            test_raw_df["phash_hex"].tolist(),
            chunk_size=512,
        )
        hit_indices = cand_indices[min_dists <= int(cfg["phash_dup_threshold"])]
        boundary_dup_mask.loc[hit_indices] = True

    excluded_mask = same_cluster_mask | boundary_dup_mask
    excluded_raw_df = work.loc[excluded_mask].copy()
    excluded_raw_df["exclusion_reason"] = np.where(
        same_cluster_mask.loc[excluded_mask],
        "same_cluster_as_locked_test",
        "cross_boundary_phash_near_dup",
    )

    pool_raw_df = work.loc[not_test_mask & (~excluded_mask)].copy()

    # 物理隔离落盘：测试集与训练/验证池写入互不相交的两个根目录
    test_df = _materialize_isolated_split(test_raw_df, clean_path / "test_locked")
    pool_df = _materialize_isolated_split(pool_raw_df, clean_path / "train_val_pool")

    classes_list = list(cfg["expected_classes"])
    class_to_idx = {c: i for i, c in enumerate(classes_list)}
    test_df["label_idx"] = test_df["label"].map(class_to_idx)
    pool_df["label_idx"] = pool_df["label"].map(class_to_idx)

    test_df.to_csv(test_csv_path, index=False)
    excluded_raw_df.to_csv(manifest_path / "excluded_near_dups_of_test.csv", index=False)
    pool_df.to_csv(manifest_path / "train_val_pool.csv", index=False)
    with open(manifest_path / "classes.json", "w", encoding="utf-8") as f:
        json.dump({"classes": classes_list, "class_to_idx": class_to_idx}, f, ensure_ascii=False, indent=2)

    lock_sha = _compute_lock_sha256(test_df)
    sha_path.write_text(lock_sha + "\n", encoding="utf-8")

    return test_df, pool_df, excluded_raw_df

In [ ]:
test_df, pool_df, excluded_near_dups_df = select_high_quality_test_and_exclude(
    valid_df,
    CFG,
    clean_root=CFG["clean_root"],
    manifest_dir=CFG["manifest_dir"],
)

print(f"锁定高质量测试集 (test_locked)      : {len(test_df)} 张")
print(f"因与测试集同簇/近重复而剔除         : {len(excluded_near_dups_df)} 张")
print(f"剩余干净训练/验证池 (train_val_pool): {len(pool_df)} 张")

summary_rows = []
for cls_name in CFG["expected_classes"]:
    t_cls = test_df[test_df["label"] == cls_name]
    p_cls = pool_df[pool_df["label"] == cls_name]
    summary_rows.append({
        "class": cls_name,
        "test_count": len(t_cls),
        "test_hq_ratio": f"{t_cls['is_high_quality'].mean():.1%}",
        "test_med_min_side": int(t_cls["min_side"].median()),
        "test_med_laplacian": round(float(t_cls["laplacian_var"].median()), 1),
        "pool_count": len(p_cls),
        "pool_med_min_side": int(p_cls["min_side"].median()),
        "pool_med_laplacian": round(float(p_cls["laplacian_var"].median()), 1),
    })
display_summary = pd.DataFrame(summary_rows)
print(display_summary.to_string(index=False))

# 抽样可视化锁定测试集（每类 5 张，共 10x5 网格）
fig, axes = plt.subplots(len(CFG["expected_classes"]), 5, figsize=(12, 22))
rng_vis = np.random.default_rng(CFG["seed"])
for r, cls_name in enumerate(CFG["expected_classes"]):
    cls_samples = test_df[test_df["label"] == cls_name]
    pick_idx = rng_vis.choice(len(cls_samples), size=min(5, len(cls_samples)), replace=False)
    for c in range(5):
        ax = axes[r, c]
        ax.axis("off")
        if c < len(pick_idx):
            row = cls_samples.iloc[pick_idx[c]]
            img = Image.open(row["isolated_path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(f"{cls_name}\n{row['width']}x{row['height']} | lap={row['laplacian_var']:.0f}", fontsize=8)
plt.suptitle("Locked High-Quality Test Set Visual Audit (5 samples / class)", fontsize=13, y=1.002)
plt.tight_layout()
plt.savefig(Path(CFG["eval_dir"]) / "test_locked_visual_audit.png", dpi=130, bbox_inches="tight")
plt.show()

## Section 5 — 在 `train_val_pool` 内分组分层切分 Train/Val + 防泄漏断言防火墙

In [ ]:
def split_train_val_grouped(pool_df: pd.DataFrame, cfg: dict, manifest_dir: str | Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split train_val_pool into train and val using StratifiedGroupKFold grouped by cluster_id."""
    sgkf = StratifiedGroupKFold(n_splits=int(cfg["val_folds"]), shuffle=True, random_state=int(cfg["seed"]))
    train_idx, val_idx = next(
        sgkf.split(pool_df, pool_df["label_idx"].to_numpy(), groups=pool_df["cluster_id"].to_numpy())
    )
    train_df = pool_df.iloc[train_idx].copy().reset_index(drop=True)
    val_df = pool_df.iloc[val_idx].copy().reset_index(drop=True)

    manifest_path = Path(manifest_dir)
    train_df.to_csv(manifest_path / "train.csv", index=False)
    val_df.to_csv(manifest_path / "val.csv", index=False)
    return train_df, val_df


def verify_no_leakage(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    clean_root: str | Path,
    phash_dup_threshold: int = 6,
) -> None:
    """Hard assertions ensuring zero path, MD5, pHash cluster, and cross-boundary leakage."""
    clean_path = Path(clean_root).resolve()
    pool_dir = (clean_path / "train_val_pool").resolve()
    test_dir = (clean_path / "test_locked").resolve()

    splits = [("train", train_df), ("val", val_df), ("test", test_df)]
    for i in range(len(splits)):
        for j in range(i + 1, len(splits)):
            name_a, df_a = splits[i]
            name_b, df_b = splits[j]
            for col in ("orig_path", "isolated_path", "md5", "cluster_id"):
                overlap = set(df_a[col]) & set(df_b[col])
                assert len(overlap) == 0, f"Leakage detected between {name_a} and {name_b} on '{col}': {len(overlap)} overlaps"

    # 物理目录边界检查
    for name, sub_df in (("train", train_df), ("val", val_df)):
        for p_str in sub_df["isolated_path"]:
            p_res = Path(p_str).resolve()
            assert p_res.is_relative_to(pool_dir), f"{name} sample outside train_val_pool: {p_res}"
            assert not p_res.is_relative_to(test_dir), f"{name} sample inside test_locked: {p_res}"

    for p_str in test_df["isolated_path"]:
        p_res = Path(p_str).resolve()
        assert p_res.is_relative_to(test_dir), f"test sample outside test_locked: {p_res}"

    # 跨边界向量化 pHash 汉明距离检查（Train+Val vs Test）
    tv_hex = pd.concat([train_df["phash_hex"], val_df["phash_hex"]], ignore_index=True).tolist()
    min_dists = min_hamming_between(tv_hex, test_df["phash_hex"].tolist(), chunk_size=512)
    if len(min_dists) > 0:
        min_overall = int(min_dists.min())
        assert min_overall > phash_dup_threshold, (
            f"Cross-boundary pHash leakage! min Hamming distance = {min_overall} <= {phash_dup_threshold}"
        )
        print(f"[Firewall Passed] Train/Val vs Test 最小跨集 pHash 汉明距离 = {min_overall} (> {phash_dup_threshold})")
    print(f"[Firewall Passed] Train={len(train_df)} | Val={len(val_df)} | Test(Locked)={len(test_df)} | 零重叠、零泄漏")


train_df, val_df = split_train_val_grouped(pool_df, CFG, CFG["manifest_dir"])
verify_no_leakage(train_df, val_df, test_df, CFG["clean_root"], CFG["phash_dup_threshold"])

## Section 6 — PyTorch Dataset、抗背景偏置增强管线与类别加权 Loss

In [ ]:
# 先实例化模型元信息以精确对齐 mean / std / input_size
_tmp_model = timm.create_model(CFG["model_name"], pretrained=False, num_classes=len(CFG["expected_classes"]))
DATA_CFG = timm.data.resolve_model_data_config(_tmp_model)
del _tmp_model

IMG_SIZE = DATA_CFG["input_size"][-2:]  # (H, W)
IMG_MEAN = list(DATA_CFG["mean"])
IMG_STD = list(DATA_CFG["std"])
print(f"Resolved model input_size={IMG_SIZE}, mean={IMG_MEAN}, std={IMG_STD}")

train_transforms = v2.Compose([
    v2.ToImage(),
    v2.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0), ratio=(0.8, 1.25), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.2),
    v2.RandomRotation(degrees=20),
    v2.RandomApply([v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.03)], p=0.4),
    v2.RandomApply([v2.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))], p=0.2),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMG_MEAN, std=IMG_STD),
    v2.RandomErasing(p=0.25, scale=(0.02, 0.15)),
])

eval_transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize(int(round(IMG_SIZE[0] / 0.875)), antialias=True),
    v2.CenterCrop(IMG_SIZE),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMG_MEAN, std=IMG_STD),
])


class GarbageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None, path_col: str = "isolated_path"):
        self.paths = df[path_col].tolist()
        self.labels = df["label_idx"].astype(int).tolist()
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        with Image.open(self.paths[idx]) as img:
            rgb = ImageOps.exif_transpose(img).convert("RGB")
        if self.transform is not None:
            tensor = self.transform(rgb)
        else:
            tensor = v2.ToDtype(torch.float32, scale=True)(v2.ToImage()(rgb))
        return tensor, self.labels[idx]


train_ds = GarbageDataset(train_df, transform=train_transforms)
val_ds = GarbageDataset(val_df, transform=eval_transforms)
test_ds = GarbageDataset(test_df, transform=eval_transforms)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

# 平方根平滑类别权重（缓解不平衡且不过度放大少数类噪声）
train_counts = train_df["label_idx"].value_counts().sort_index().to_numpy(dtype=np.float32)
raw_weights = (train_counts.sum() / np.maximum(train_counts, 1.0)) ** 0.5
class_weights = torch.tensor(raw_weights / raw_weights.mean(), dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CFG["label_smoothing"])
print("Smoothed class weights:", dict(zip(CFG["expected_classes"], np.round(class_weights.cpu().numpy(), 3))))

# 可视化一个增强 Batch（8 张）
mean_t = torch.tensor(IMG_MEAN).view(3, 1, 1)
std_t = torch.tensor(IMG_STD).view(3, 1, 1)
xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for idx, ax in enumerate(axes.flat):
    vis = (xb[idx] * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(vis)
    ax.set_title(CFG["expected_classes"][int(yb[idx])], fontsize=10)
    ax.axis("off")
plt.suptitle("Augmented Training Batch Preview", fontsize=12)
plt.tight_layout()
plt.show()

## Section 7 — `timm` 模型构建与两阶段迁移学习（AMP + 断点续训 + 早停）

In [ ]:
def set_backbone_trainable(model: nn.Module, trainable: bool) -> tuple[list[nn.Parameter], list[nn.Parameter]]:
    """Freeze or unfreeze backbone while keeping the classifier head trainable."""
    classifier = model.get_classifier() if hasattr(model, "get_classifier") else None
    head_param_ids: set[int] = set()
    if isinstance(classifier, nn.Module):
        head_param_ids = {id(p) for p in classifier.parameters()}
    else:
        for name, param in model.named_parameters():
            if any(k in name for k in ("head", "fc", "classifier")):
                head_param_ids.add(id(param))

    backbone_params, head_params = [], []
    for param in model.parameters():
        if id(param) in head_param_ids:
            param.requires_grad = True
            head_params.append(param)
        else:
            param.requires_grad = trainable
            backbone_params.append(param)
    return backbone_params, head_params


def evaluate_loader(model: nn.Module, loader: DataLoader, loss_fn: nn.Module, amp_dtype, tta_hflip: bool = False):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_probs, all_targets = [], []

    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=amp_dtype):
                logits = model(images)
                if tta_hflip:
                    logits = 0.5 * (logits + model(torch.flip(images, dims=[3])))
                loss = loss_fn(logits, targets)

            bs = targets.size(0)
            total_loss += float(loss.item()) * bs
            total_samples += bs
            all_probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    probs = np.concatenate(all_probs, axis=0)
    y_true = np.concatenate(all_targets, axis=0)
    y_pred = probs.argmax(axis=1)
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))
    return {
        "loss": total_loss / max(total_samples, 1),
        "acc": acc,
        "macro_f1": macro_f1,
        "probs": probs,
        "y_true": y_true,
        "y_pred": y_pred,
    }


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    scaler: torch.amp.GradScaler,
    amp_dtype,
    grad_accum_steps: int = 1,
    desc: str = "Train",
) -> float:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    total_samples = 0

    pbar = tqdm(enumerate(loader), total=len(loader), desc=desc, leave=False)
    for step, (images, targets) in pbar:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=amp_dtype):
            logits = model(images)
            raw_loss = loss_fn(logits, targets)
            scaled_loss = raw_loss / grad_accum_steps

        scaler.scale(scaled_loss).backward()

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        bs = targets.size(0)
        running_loss += float(raw_loss.item()) * bs
        total_samples += bs
        pbar.set_postfix(loss=f"{running_loss / total_samples:.4f}")

    return running_loss / max(total_samples, 1)


model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=len(CFG["expected_classes"]),
    drop_path_rate=CFG["drop_path_rate"],
).to(DEVICE)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if use_bf16 else torch.float16
USE_FP16_SCALER = torch.cuda.is_available() and (not use_bf16)
scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16_SCALER)

last_ckpt_path = Path(CFG["ckpt_dir"]) / "last.pt"
best_ckpt_path = Path(CFG["ckpt_dir"]) / "best.pt"
history_csv_path = Path(CFG["ckpt_dir"]) / "history.csv"

total_epochs = CFG["stage1_epochs"] + CFG["stage2_epochs"]
start_epoch = 0
best_val_f1 = -1.0
no_improve_epochs = 0
history: list[dict] = []

# 支持断线续训
if last_ckpt_path.exists():
    ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)
    if ckpt.get("model_name") == CFG["model_name"]:
        model.load_state_dict(ckpt["model_state"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_val_f1 = float(ckpt["best_val_f1"])
        no_improve_epochs = int(ckpt.get("no_improve_epochs", 0))
        history = list(ckpt.get("history", []))
        print(f"[Resume] 从 {last_ckpt_path} 恢复：已完成 {start_epoch}/{total_epochs} epochs，当前最佳 Val Macro-F1 = {best_val_f1:.4f}")

# 构建 Stage1 与 Stage2 优化器/调度器（若断点在 Stage2 中段则快速重建并同步步数）
stage1_opt = torch.optim.AdamW(set_backbone_trainable(model, False)[1], lr=CFG["stage1_lr"], weight_decay=CFG["weight_decay"])

backbone_p, head_p = set_backbone_trainable(model, True)
stage2_opt = torch.optim.AdamW(
    [
        {"params": backbone_p, "lr": CFG["backbone_lr"]},
        {"params": head_p, "lr": CFG["head_lr"]},
    ],
    weight_decay=CFG["weight_decay"],
)
warmup_sch = torch.optim.lr_scheduler.LinearLR(stage2_opt, start_factor=0.1, total_iters=1)
cosine_sch = torch.optim.lr_scheduler.CosineAnnealingLR(
    stage2_opt, T_max=max(1, CFG["stage2_epochs"] - 1), eta_min=1e-6
)
stage2_sch = torch.optim.lr_scheduler.SequentialLR(stage2_opt, schedulers=[warmup_sch, cosine_sch], milestones=[1])

if start_epoch > CFG["stage1_epochs"] and last_ckpt_path.exists():
    ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)
    if ckpt.get("stage") == 2 and "optimizer_state" in ckpt:
        stage2_opt.load_state_dict(ckpt["optimizer_state"])
    if ckpt.get("stage") == 2 and "scheduler_state" in ckpt and ckpt["scheduler_state"] is not None:
        stage2_sch.load_state_dict(ckpt["scheduler_state"])
elif 0 < start_epoch <= CFG["stage1_epochs"] and last_ckpt_path.exists():
    ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)
    if ckpt.get("stage") == 1 and "optimizer_state" in ckpt:
        stage1_opt.load_state_dict(ckpt["optimizer_state"])

for epoch in range(start_epoch, total_epochs):
    if epoch < CFG["stage1_epochs"]:
        stage = 1
        set_backbone_trainable(model, False)
        active_opt, active_sch = stage1_opt, None
    else:
        stage = 2
        set_backbone_trainable(model, True)
        active_opt, active_sch = stage2_opt, stage2_sch

    train_loss = train_one_epoch(
        model,
        train_loader,
        active_opt,
        criterion,
        scaler,
        amp_dtype=AMP_DTYPE,
        grad_accum_steps=CFG["grad_accum_steps"],
        desc=f"Epoch {epoch + 1}/{total_epochs} [Stage {stage}]",
    )
    if active_sch is not None:
        active_sch.step()

    val_res = evaluate_loader(model, val_loader, criterion, amp_dtype=AMP_DTYPE, tta_hflip=False)
    row = {
        "epoch": epoch + 1,
        "stage": stage,
        "train_loss": train_loss,
        "val_loss": val_res["loss"],
        "val_acc": val_res["acc"],
        "val_macro_f1": val_res["macro_f1"],
    }
    history.append(row)
    pd.DataFrame(history).to_csv(history_csv_path, index=False)

    is_best = val_res["macro_f1"] > best_val_f1
    if is_best:
        best_val_f1 = val_res["macro_f1"]
        no_improve_epochs = 0
        torch.save(
            {
                "epoch": epoch,
                "model_name": CFG["model_name"],
                "classes": CFG["expected_classes"],
                "model_state": model.state_dict(),
                "best_val_f1": best_val_f1,
                "img_size": IMG_SIZE,
                "mean": IMG_MEAN,
                "std": IMG_STD,
            },
            best_ckpt_path,
        )
    elif stage == 2:
        no_improve_epochs += 1

    torch.save(
        {
            "epoch": epoch,
            "stage": stage,
            "model_name": CFG["model_name"],
            "model_state": model.state_dict(),
            "optimizer_state": active_opt.state_dict(),
            "scheduler_state": active_sch.state_dict() if active_sch is not None else None,
            "best_val_f1": best_val_f1,
            "no_improve_epochs": no_improve_epochs,
            "history": history,
        },
        last_ckpt_path,
    )

    mark = " *BEST*" if is_best else ""
    print(
        f"Epoch {epoch + 1:02d}/{total_epochs} (S{stage}) | "
        f"TrainLoss={train_loss:.4f} | ValLoss={val_res['loss']:.4f} | "
        f"ValAcc={val_res['acc']:.4f} | ValMacroF1={val_res['macro_f1']:.4f}{mark}"
    )

    if stage == 2 and no_improve_epochs >= CFG["early_stop_patience"]:
        print(f"[Early Stop] Stage 2 连续 {CFG['early_stop_patience']} 个 epoch Val Macro-F1 未提升，提前结束训练。")
        break

# 绘制训练曲线
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="Train Loss")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"], label="Val Loss")
axes[0].axvline(CFG["stage1_epochs"] + 0.5, color="gray", linestyle="--", label="Stage 2 Start")
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(hist_df["epoch"], hist_df["val_acc"], label="Val Acc")
axes[1].plot(hist_df["epoch"], hist_df["val_macro_f1"], label="Val Macro-F1")
axes[1].axvline(CFG["stage1_epochs"] + 0.5, color="gray", linestyle="--", label="Stage 2 Start")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(Path(CFG["eval_dir"]) / "training_curves.png", dpi=150)
plt.show()

## Section 8 — 锁定测试集最终评测、2-View TTA、错例网格与 Grad-CAM 可解释性

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

best_ckpt = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt["model_state"])
model.eval()

single_res = evaluate_loader(model, test_loader, criterion, amp_dtype=AMP_DTYPE, tta_hflip=False)
tta_res = evaluate_loader(model, test_loader, criterion, amp_dtype=AMP_DTYPE, tta_hflip=True)

print("=== Locked High-Quality Test Set Evaluation ===")
print(f"Single-View | Acc={single_res['acc']:.4f} | Macro-F1={single_res['macro_f1']:.4f}")
print(f"2-View TTA  | Acc={tta_res['acc']:.4f} | Macro-F1={tta_res['macro_f1']:.4f}\n")
print(classification_report(tta_res["y_true"], tta_res["y_pred"], target_names=CFG["expected_classes"], digits=4))

# 归一化混淆矩阵
cm = confusion_matrix(tta_res["y_true"], tta_res["y_pred"], normalize="true")
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=CFG["expected_classes"],
    yticklabels=CFG["expected_classes"],
    ax=ax,
)
ax.set_title("Locked Test Set Normalized Confusion Matrix (2-View TTA)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(Path(CFG["eval_dir"]) / "test_confusion_matrix.png", dpi=150)
plt.show()

# Top-16 高置信错例网格
probs = tta_res["probs"]
y_true = tta_res["y_true"]
y_pred = tta_res["y_pred"]
wrong_idx = np.where(y_true != y_pred)[0]
if len(wrong_idx) > 0:
    conf_wrong = probs[wrong_idx, y_pred[wrong_idx]]
    top_wrong = wrong_idx[np.argsort(-conf_wrong)[:16]]
    n_show = len(top_wrong)
    cols = 4
    rows = math.ceil(n_show / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(13, 3.2 * rows))
    axes_flat = np.array(axes).reshape(-1)
    for k, ax in enumerate(axes_flat):
        ax.axis("off")
        if k < n_show:
            idx = int(top_wrong[k])
            row_meta = test_df.iloc[idx]
            img = Image.open(row_meta["isolated_path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(
                f"T: {CFG['expected_classes'][y_true[idx]]}\n"
                f"P: {CFG['expected_classes'][y_pred[idx]]} ({probs[idx, y_pred[idx]]:.2f})",
                fontsize=9,
                color="crimson",
            )
    plt.suptitle("Top High-Confidence Misclassified Test Samples", fontsize=12)
    plt.tight_layout()
    plt.savefig(Path(CFG["eval_dir"]) / "test_top_errors.png", dpi=140)
    plt.show()
else:
    print("测试集无错分样本！")


# Grad-CAM 定位最后一层特征图（兼容 ConvNeXt / EfficientNet / ResNet）
def find_gradcam_target_layer(net: nn.Module) -> nn.Module:
    if hasattr(net, "stages") and len(net.stages) > 0:
        last_stage = net.stages[-1]
        if hasattr(last_stage, "blocks") and len(last_stage.blocks) > 0:
            return last_stage.blocks[-1]
        return last_stage
    if hasattr(net, "conv_head"):
        return net.conv_head
    last_conv = None
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            last_conv = m
    if last_conv is None:
        raise ValueError("未找到可用于 Grad-CAM 的卷积层")
    return last_conv


target_layer = find_gradcam_target_layer(model)
cam = GradCAM(model=model, target_layers=[target_layer])
rng_cam = np.random.default_rng(CFG["seed"])
sample_cam_idx = rng_cam.choice(len(test_df), size=min(8, len(test_df)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, idx in zip(axes.flat, sample_cam_idx):
    row_meta = test_df.iloc[int(idx)]
    with Image.open(row_meta["isolated_path"]) as im:
        rgb = ImageOps.exif_transpose(im).convert("RGB")
    input_tensor = eval_transforms(rgb).unsqueeze(0).to(DEVICE)

    vis_img = rgb.resize(IMG_SIZE[::-1])
    vis_np = np.asarray(vis_img, dtype=np.float32) / 255.0

    pred_cls = int(y_pred[int(idx)])
    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_cls)])[0]
    overlay = show_cam_on_image(vis_np, grayscale_cam, use_rgb=True)
    ax.imshow(overlay)
    ax.set_title(
        f"T: {row_meta['label']} | P: {CFG['expected_classes'][pred_cls]}",
        fontsize=9,
    )
    ax.axis("off")
plt.suptitle("Grad-CAM Saliency Maps on Locked Test Samples", fontsize=12)
plt.tight_layout()
plt.savefig(Path(CFG["eval_dir"]) / "test_gradcam_samples.png", dpi=140)
plt.show()
del cam

## Section 9 — 可选外部 OOD 零样本泛化摸底（`RealWaste` + 手机实拍图）

这一步**不参与任何训练**，仅用作“照妖镜”来量化 `v2` 单库模型在真实垃圾场（`RealWaste`）或你手机实拍照片上的域偏移掉了多少分，帮你客观判断后续是否需要进入 Step 2 联合训练。

In [ ]:
import urllib.request

RUN_REALWASTE_OOD_EVAL = True

REALWASTE_TO_V2 = {
    "Cardboard": "cardboard",
    "Food Organics": "biological",
    "Vegetation": "biological",
    "Glass": "glass",
    "Metal": "metal",
    "Miscellaneous Trash": "trash",
    "Paper": "paper",
    "Plastic": "plastic",
    "Textile Trash": "clothes",
}


def download_realwaste_root(target_dir: Path) -> Path | None:
    target_dir.mkdir(parents=True, exist_ok=True)
    # 先尝试通过 KaggleHub 获取，其次回退 UCI 官方 ZIP
    try:
        rw_path = Path(kagglehub.dataset_download("joarvidsson/realwaste-image-classification"))
        return locate_class_root(rw_path, list(REALWASTE_TO_V2.keys()))
    except Exception as exc:
        print(f"[Info] KaggleHub 拉取 RealWaste 失败，尝试 UCI 官方 ZIP: {exc}")

    zip_path = target_dir / "realwaste.zip"
    extract_dir = target_dir / "extracted"
    if not extract_dir.exists():
        try:
            urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/908/realwaste.zip", zip_path)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(extract_dir)
        except Exception as exc:
            print(f"[Warning] RealWaste 自动下载失败（跳过 OOD 评测）: {exc}")
            return None
    try:
        return locate_class_root(extract_dir, list(REALWASTE_TO_V2.keys()))
    except Exception as exc:
        print(f"[Warning] 未能解析 RealWaste 目录结构: {exc}")
        return None


if RUN_REALWASTE_OOD_EVAL:
    rw_root = download_realwaste_root(BASE_WORK_DIR / "realwaste_ood")
    if rw_root is not None:
        rw_records = []
        class_to_idx = {c: i for i, c in enumerate(CFG["expected_classes"])}
        for rw_cls, v2_cls in REALWASTE_TO_V2.items():
            cls_dir = rw_root / rw_cls
            if not cls_dir.is_dir():
                continue
            for p in cls_dir.rglob("*"):
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    rw_records.append({
                        "isolated_path": str(p),
                        "label": v2_cls,
                        "label_idx": class_to_idx[v2_cls],
                    })
        rw_df = pd.DataFrame(rw_records)
        if len(rw_df) > 0:
            rw_loader = DataLoader(
                GarbageDataset(rw_df, transform=eval_transforms),
                batch_size=CFG["batch_size"],
                shuffle=False,
                num_workers=CFG["num_workers"],
            )
            rw_res = evaluate_loader(model, rw_loader, criterion, amp_dtype=AMP_DTYPE, tta_hflip=True)
            overlapping_labels = sorted(rw_df["label_idx"].unique().tolist())
            overlapping_names = [CFG["expected_classes"][i] for i in overlapping_labels]
            rw_f1 = f1_score(rw_res["y_true"], rw_res["y_pred"], labels=overlapping_labels, average="macro")
            print(f"=== RealWaste Zero-Shot OOD ({len(rw_df)} images, 8 overlapping classes) ===")
            print(f"OOD Acc={rw_res['acc']:.4f} | Overlapping Macro-F1={rw_f1:.4f}")
            print(
                classification_report(
                    rw_res["y_true"],
                    rw_res["y_pred"],
                    labels=overlapping_labels,
                    target_names=overlapping_names,
                    digits=4,
                    zero_division=0,
                )
            )

# 可选：评估用户放在 Google Drive my_real_photos/<class>/ 下的手机实拍图
real_dir = Path(CFG["real_photos_dir"])
custom_records = []
class_to_idx = {c: i for i, c in enumerate(CFG["expected_classes"])}
for cls_name, cls_idx in class_to_idx.items():
    c_dir = real_dir / cls_name
    if c_dir.is_dir():
        for p in c_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                custom_records.append({"isolated_path": str(p), "label": cls_name, "label_idx": cls_idx})

if len(custom_records) > 0:
    custom_df = pd.DataFrame(custom_records)
    custom_loader = DataLoader(GarbageDataset(custom_df, transform=eval_transforms), batch_size=16, shuffle=False)
    custom_res = evaluate_loader(model, custom_loader, criterion, amp_dtype=AMP_DTYPE, tta_hflip=True)
    print(f"=== Custom Phone Photos Eval ({len(custom_df)} images) ===")
    print(f"Acc={custom_res['acc']:.4f} | Macro-F1={custom_res['macro_f1']:.4f}")
else:
    print(f"[Tip] 如需检验真实泛化，可将手机实拍图按类别放入 {real_dir}/<class>/ 后重新运行本 Cell。")

## Section 10 — Gradio 交互式 Web 演示（含环保四分类映射与低置信拒识）

In [ ]:
import gradio as gr

RECYCLING_GUIDE = {
    "battery": ("有害垃圾", "请单独投放至有害垃圾收集容器，切勿破损或混入生活垃圾。"),
    "biological": ("厨余垃圾", "沥干水分后投入厨余垃圾桶（硬骨、大贝壳等通常归入其他垃圾）。"),
    "cardboard": ("可回收物", "压扁保持干燥清洁后投放入可回收物箱。"),
    "clothes": ("可回收物", "干净干燥的废旧衣物可投放至旧衣回收箱；严重污损请按其他垃圾处理。"),
    "glass": ("可回收物", "倒空内容物后轻投至可回收物箱，碎玻璃请包裹防割伤。"),
    "metal": ("可回收物", "压扁金属罐并排空残留液体后投放可回收物箱。"),
    "paper": ("可回收物", "干燥无油污废纸为可回收物；沾染油污/水渍的餐巾纸请投其他垃圾。"),
    "plastic": ("可回收物", "清洁干燥塑料瓶/包装归可回收物；严重油污塑料袋建议投其他垃圾。"),
    "shoes": ("可回收物", "成双干净旧鞋可投旧衣鞋回收箱；严重破损脏污按其他垃圾处理。"),
    "trash": ("其他垃圾", "难以归类或受污染的杂物请投入其他垃圾桶。"),
}


@torch.inference_mode()
def predict_garbage(pil_img: Image.Image):
    if pil_img is None:
        return {}, "请先上传或拍摄单件垃圾图片。"

    rgb = ImageOps.exif_transpose(pil_img).convert("RGB")
    tensor = eval_transforms(rgb).unsqueeze(0).to(DEVICE)
    with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=AMP_DTYPE):
        logits = 0.5 * (model(tensor) + model(torch.flip(tensor, dims=[3])))
    probs = torch.softmax(logits.float(), dim=1).squeeze(0).cpu().numpy()

    prob_dict = {cls_name: float(probs[i]) for i, cls_name in enumerate(CFG["expected_classes"])}
    top_idx = int(probs.argmax())
    top_cls = CFG["expected_classes"][top_idx]
    top_conf = float(probs[top_idx])

    if top_conf < CFG["ood_conf_threshold"]:
        advice = (
            f"置信度偏低（Top-1: `{top_cls}` 仅 {top_conf:.1%} < 阈值 {CFG['ood_conf_threshold']:.0%}）。\n\n"
            "难以确认该物品类别：请靠近对准**单件垃圾物体**拍摄，或确认图片中确实为垃圾物品。"
        )
    else:
        cat_name, tip = RECYCLING_GUIDE[top_cls]
        advice = (
            f"### 识别结果：`{top_cls}`（置信度 **{top_conf:.1%}**）\n"
            f"- **建议分类**：**{cat_name}**\n"
            f"- **投放提示**：{tip}"
        )
    return prob_dict, advice


demo = gr.Interface(
    fn=predict_garbage,
    inputs=gr.Image(type="pil", label="上传或拍摄单件垃圾图片"),
    outputs=[
        gr.Label(num_top_classes=5, label="Top-5 类别概率"),
        gr.Markdown(),
    ],
    title="Garbage Classification v2 — Step 1 模型演示",
    description="基于 ConvNeXt-Tiny + 高质量防泄漏测试集训练。注：单标签模型适合单件物品识别，混合多目标垃圾桶请勿直接拍照识别。",
)

demo.launch(share=True, debug=False)